#  CUDA Programming
# 1. Introduction
* [CUDA](https://docs.nvidia.com/cuda/index.html) is an extension to C++ specific to NVIDIA GPUs
* It is the most basic, native programming model for NVIDIA GPUs
* See [CUDA C++ Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html) for reference

# 2. How to work on this notebook
* Read text, modify cells if necessary, and execute them
* Many cells can be executed without modifying them
* Cells that need your work are typically preceeded with <font color="green">text colored green explaining what you need to do</font>
* Problems with non-trivial work have a section header starting with <font color="green">"Problem: X"</font>

# 3. Jupyter AI
* You can ask Jupyter AI when you need a help
* Open `pd04_cuda.chat` in this folder
* Choose `OpenCode` in Persona Chooser
* You can try several models, but `LiteLLM (taulec)/GPT-5.3 Codex (UTokyo Azure)` is recommended for now
* It is instructed to help you but not to give you answer directly to exercise problems; do not try to circumvent it

# 4. Setting up compilers
* We use [NVIDIA CUDA Toolkit](https://developer.nvidia.com/cuda-toolkit) (`nvcc`) as our CUDA compiler
* NVIDIA's C/C++ compilers (`nvc` and `nvc++`) and [LLVM](https://llvm.org/) (`clang` and `clang++`) we used for OpenMP also support CUDA, but they fail to compile some of our code, so we stick to more traditional `nvcc`
* As in the OpenMP notebook, run this setup at the start of a `%%bash` cell. It makes all of them available:


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH

* Check if it works


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
which nvcc
which nvc++
which clang++
nvcc --version

- If you prefer to work in a terminal, run the above `module` and `PATH` commands in each new shell, or add them to a Miyabi shell startup file such as `~/.bashrc`.

# 5. Check the GPU
* Check if GPU is alive by `nvidia-smi`
* Make sure you execute `%load_ext miyabi` cell before executing `%%sub` cell
* Do `nvidia-smi --help` or see manual (`man nvidia-smi` on terminal) for more info


In [ ]:
%load_ext miyabi

In [ ]:
%%sub
nvidia-smi

* Remember that login node does _not_ have GPU, so the following does _not_ work


In [ ]:
%%bash
# this does *not* work
nvidia-smi

# 6. Compiling and running CUDA programs
* Give a source file `.cu` extension or give `--x cu` option to indicate it is a CUDA source file


In [ ]:
%%writefile cuda_hello.cu
#include <assert.h>
#include <stdio.h>

__global__ void cuda_thread_fun(int n) {
  int i        = blockDim.x * blockIdx.x + threadIdx.x;
  int nthreads = gridDim.x * blockDim.x;
  if (i < n) {
    printf("hello I am CUDA thread %d out of %d\n", i, nthreads);
  }
}

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 100);
  int thread_block_sz = (argc > 2 ? atoi(argv[2]) : 64);
  int n_thread_blocks = (n + thread_block_sz - 1) / thread_block_sz;
  printf("%d threads/block * %d blocks\n", thread_block_sz, n_thread_blocks);

  // launch a kernel
  cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(n);
  // wait for them to complete
  cudaDeviceSynchronize();
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_hello cuda_hello.cu

In [ ]:
%%sub
./cuda_hello

* You should see 100 lines of "hello I am CUDA thread ??? out of 128"

* Alternatively, you can have a source file with an ordinary C++ extension `.cc` (or `.cpp`) and give `-x cu`.
* It is useful when you want to have a single source file for OpenMP and CUDA programs


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
cp cuda_hello.cu cuda_hello.cc
nvcc -arch=sm_90 -o cuda_hello -x cu cuda_hello.cc

In [ ]:
%%sub
./cuda_hello

# 7. CUDA kernel
* The most basic concept of CUDA programming is a _CUDA kernel_
* Syntactically, a CUDA kernel is a `void` function with `__global__` keyword attached to it
```
__global__ void cuda_thread_fun(int n) { ... }
```
* A CUDA kernel describes what a _single_ CUDA thread does
* You launch a number of CUDA threads all executing the same kernel by 
```
kernel_func<<<num_of_blocks,num_of_threads_per_block>>>(...);
```
* We have already seen this in the above code
* See [2. Programming model](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#programming-model) section for reference

## 7-1. <font color="red">You'd better always check errors</font>
* It's not only CUDA programming in which you are strongly advised to check errors after each operation that could potentially go wrong
* Just like many C programming APIs (unlike Python scripting, for example), calling CUDA APIs and launching CUDA kernels silently return if something went wrong
* You could save a huge amount of time by checking errors 
  * every time you launch a CUDA kernel and
  * every time you call a CUDA API

* Remember, the following does _not_ work
* Worse, no error is signaled; both kernel launch and API call _silentely_ fail


In [ ]:
%%bash
./cuda_hello

* Here is the same piece of code with checking errors


In [ ]:
%%writefile cuda_hello_chk.cu
#include <assert.h>
#include <stdio.h>

/*
  you'd better spend time on making sure you always check errors ...
*/

void check_api_error_(cudaError_t e,
                      const char * msg, const char * file, int line) {
  if (e) {
    fprintf(stderr, "%s:%d:error: %s %s\n",
            file, line, msg, cudaGetErrorString(e));
    exit(1);
  }
}

#define check_api_error(e) check_api_error_(e, #e, __FILE__, __LINE__)

void check_launch_error_(const char * msg, const char * file, int line) {
  cudaError_t e = cudaGetLastError();
  if (e) {
    fprintf(stderr, "%s:%d:error: %s %s\n",
            file, line, msg, cudaGetErrorString(e));
    exit(1);
  }
}

#define check_launch_error(exp) do { exp; check_launch_error_(#exp, __FILE__, __LINE__); } while (0)


__global__ void cuda_thread_fun(int n) {
  int i        = blockDim.x * blockIdx.x + threadIdx.x;
  int nthreads = gridDim.x * blockDim.x;
  if (i < n) {
    printf("hello I am CUDA thread %d out of %d\n", i, nthreads);
  }
}

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 100);
  int thread_block_sz = (argc > 2 ? atoi(argv[2]) : 64);
  int n_thread_blocks = (n + thread_block_sz - 1) / thread_block_sz;
  printf("%d threads/block * %d blocks\n", thread_block_sz, n_thread_blocks);

  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(n)));
  check_api_error(cudaDeviceSynchronize());
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_hello_chk cuda_hello_chk.cu

* Let's witness it catches the error failing to launch a kernel


In [ ]:
%%bash
./cuda_hello_chk

* Let's factor out the error-checking code into a header file `"cuda_util.h"` and reuse it down the road


In [ ]:
%%writefile cuda_util.h
/**
   @file cuda_util.h
   @brief small utility functions for cuda
   @author Kenjiro Taura
   @date Oct. 14, 2018
 */

/**
   @brief do not use this function directly. use check_api_error macro
   @sa check_api_error
 */

static void check_api_error_(cudaError_t e,
                              const char * msg, const char * file, int line) {
  if (e) {
    fprintf(stderr, "%s:%d:error: %s %s\n",
            file, line, msg, cudaGetErrorString(e));
    exit(1);
  }
}

/**
   @brief check if a CUDA API invocation succeeded and show the error msg if any
   @details usage:  check_api_error(cuda_api_call()). for example,
   check_api_error(cudaMalloc(&p, size));
 */

#define check_api_error(e) check_api_error_(e, #e, __FILE__, __LINE__)

/**
   @brief do not use this function directly. use check_launch_error macro
   @sa check_launch_error
 */

static void check_launch_error_(const char * msg, const char * file, int line) {
  cudaError_t e = cudaGetLastError();
  if (e) {
    fprintf(stderr, "%s:%d:error: %s %s\n",
            file, line, msg, cudaGetErrorString(e));
    exit(1);
  }
}

/**
   @brief check kernel launch error
   @details usage: check_launch_error((kernel-launch-expression)). for example,
   check_launch_error((your_gpu_kernel<<<n_blocks,block_sz>>>(a,b,c))). 
   note that you need to put parens around the expression.
 */

#define check_launch_error(exp) do { exp; check_launch_error_(#exp, __FILE__, __LINE__); } while (0)

/**
   @brief get SM executing the caller
 */
__device__ uint get_smid(void) {
  uint ret;
  asm("mov.u32 %0, %%smid;" : "=r"(ret) );
  return ret;
}

/**
   @brief get device frequency
 */
__attribute__((unused))
static int get_freq() {
  struct cudaDeviceProp prop[1];
  check_api_error(cudaGetDeviceProperties(prop, 0));
  return prop->clockRate;
}

/**
   @brief wrap cudaMalloc.  cudaMalloc + error check + more ordinary malloc-like interface (return pointer)
 */
__attribute__((unused))
static void * dev_malloc(size_t sz) {
  void * a = 0;
  cudaError_t e = cudaMalloc(&a, sz);
  if (!a) {
    fprintf(stderr, "error: %s\n", cudaGetErrorString(e));
    exit(1);
  }
  return a;
}

/**
   @brief wrap cudaFree
 */
__attribute__((unused))
static void dev_free(void * a) {
  cudaFree(a);
}

/**
   @brief wrap cudaMemcpy to copy from device to host (and check an error if any)
 */
__attribute__((unused))
static void to_host(void * dst, void * src, size_t sz) {
  check_api_error(cudaMemcpy(dst, src, sz, cudaMemcpyDeviceToHost));
}

/**
   @brief wrap cudaMemcpy to copy from host to device (and check an error if any)
 */
__attribute__((unused))
static void to_dev(void * dst, void * src, size_t sz) {
  check_api_error(cudaMemcpy(dst, src, sz, cudaMemcpyHostToDevice));
}

/**
   @brief thread ID along x-dimension
 */
__device__ inline int get_thread_id_x() {
  return blockDim.x * blockIdx.x + threadIdx.x;
}

/**
   @brief thread ID along y-dimension
 */
__device__ inline int get_thread_id_y() {
  return blockDim.y * blockIdx.y + threadIdx.y;
}

/**
   @brief thread ID along z-dimension
 */
__device__ inline int get_thread_id_z() {
  return blockDim.z * blockIdx.z + threadIdx.z;
}

/**
   @brief number of threads along x-dimension
 */
__device__ inline int get_nthreads_x() {
  return gridDim.x * blockDim.x;
}

/**
   @brief number of threads along y-dimension
 */
__device__ inline int get_nthreads_y() {
  return gridDim.y * blockDim.y;
}

/**
   @brief number of threads along z-dimension
 */
__device__ inline int get_nthreads_z() {
  return gridDim.z * blockDim.z;
}

/**
   @brief global (x,y,z combined into an integer) thread ID
 */
__device__ inline int get_thread_id() {
  int x = get_thread_id_x();
  int y = get_thread_id_y();
  int z = get_thread_id_z();
  int nx = get_nthreads_x();
  int ny = get_nthreads_y();
  return nx * ny * z + nx * y + x;
}

/**
   @brief total number of threads
 */
__device__ inline int get_nthreads() {
  int nx = get_nthreads_x();
  int ny = get_nthreads_y();
  int nz = get_nthreads_z();
  return nx * ny * nz;
}



* Then we can make the code more concise


In [ ]:
%%writefile cuda_hello_hdr_chk.cu
#include <assert.h>
#include <stdio.h>

#include "cuda_util.h"

__global__ void cuda_thread_fun(int n) {
  int i        = blockDim.x * blockIdx.x + threadIdx.x;
  int nthreads = gridDim.x * blockDim.x;
  if (i < n) {
    printf("hello I am CUDA thread %d out of %d\n", i, nthreads);
  }
}

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 100);
  int thread_block_sz = (argc > 2 ? atoi(argv[2]) : 64);
  int n_thread_blocks = (n + thread_block_sz - 1) / thread_block_sz;
  printf("%d threads/block * %d blocks\n", thread_block_sz, n_thread_blocks);

  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(n)));
  check_api_error(cudaDeviceSynchronize());
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_hello_hdr_chk cuda_hello_hdr_chk.cu

* An error should be caught below


In [ ]:
%%bash
./cuda_hello_hdr_chk

* This one should succeed


In [ ]:
%%sub
./cuda_hello_hdr_chk

# 8. The number of CUDA threads launched
* You specify the number of threads launched by the two parameters in <<<...,...>>>, like
```
kernel_func<<<num_of_blocks,num_of_threads_per_block>>>(...);
```

* It will create (num_of_blocks * num_of_threads_per_block) threads in total.
* More precisely, it creates num_of_blocks _thread blocks_, each of which has num_of_threads_per_block threads.
* It is natural to wonder why you need to specify two parameters instead of just one parameter (the total number of threads) and how to choose num_of_threads_per_block.
* For now, just know that a thread block is the unit of scheduling
  * A GPU device fetches a single block at a time and dispatches it to a particular streaming multiprocessor (SM)
  * Remember that a single SM is like a CPU core; a single GPU device has a number of SMs just like a single CPU has a number of cores.

<font color="green">Exercise: Change the arguments of the following command line in various ways and see what happens</font>


In [ ]:
%%sub
./cuda_hello_hdr_chk 10 3

# 9. Thread ID
## 9-1. One-dimensional ID
* Just like OpenMP, CUDA provides a means for a thread to know its ID as well as the total number of threads launched together
* They are obtained from builtin variables
* Let's say you invoked a kernel with
```
f<<<12,34>>>(...);
```
you create 12 thread blocks having 34 threads each (408 threads in total).
 * `gridDim.x` gives the number of thread blocks ($= 12$)
 * `blockDim.x` gives the number of threads in a thread block ($= 34$)

* note: "grid" is the CUDA terminology to mean all the launched thread blocks (a CUDA thread $\in$ thread block $\in$ the entire grid)

 * `blockIdx.x` gives the block ID within the grid ($\in [0,12)$) 
 * `threadIdx.x` gives the thread ID within a thread block ($\in [0,34)$)
* If you want to get a single thread ID between 0 to 407 and the total number of threads, you get them by
```
int idx      = blockIdx.x * blockDim.x + threadIdx.x;
int nthreads = gridDim.x * blockDim.x;
```
* You have seen them in the above example.

* See [2.1 Kernels](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#kernels) for reference

## 9-2. Two- or three-dimensional ID
* Each of the above four variables can actually have up to three elements, allowing you to view blocks and threads within a block arranged in an one-, two- or three-dimensional space.  
* You specify them accordingly when you call a kernel, for which you use a variable of type `dim3` instead of an integer, to specify up to three numbers

* See [2.2 Thread Hierarchy](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#thread-hierarchy) for reference



In [ ]:
%%writefile cuda_hello_2d.cu
#include <assert.h>
#include <stdio.h>

#include "cuda_util.h"

__global__ void cuda_thread_fun(int n) {
  int x          = blockDim.x * blockIdx.x + threadIdx.x;
  int y          = blockDim.y * blockIdx.y + threadIdx.y;
  int nthreads_x = gridDim.x * blockDim.x;
  int nthreads_y = gridDim.y * blockDim.y;
  int g          = x + nthreads_y * y;
  if (g < n) {
    printf("hello I am CUDA thread (%d,%d) of (%d,%d)\n",
           x, y, nthreads_x, nthreads_y);
  }
}

int isqrt(int n) {
  int i;
  for (i = 0; i * i < n; i++) ;
  return i;
}

int main(int argc, char ** argv) {
  int n                 = (argc > 1 ? atoi(argv[1]) : 40);
  int nx                = isqrt(n);
  int ny                = (n + nx - 1) / nx;
  int thread_block_sz_x = (argc > 2 ? atoi(argv[2]) : 2);
  int thread_block_sz_y = (argc > 3 ? atoi(argv[3]) : 3);
  int n_thread_blocks_x = (nx + thread_block_sz_x - 1) / thread_block_sz_x;
  int n_thread_blocks_y = (ny + thread_block_sz_y - 1) / thread_block_sz_y;
  printf("(%d * %d) threads/block * (%d * %d) blocks\n",
         thread_block_sz_x, thread_block_sz_y,
         n_thread_blocks_x, n_thread_blocks_y);

  dim3 nb(n_thread_blocks_x, n_thread_blocks_y);
  dim3 tpb(thread_block_sz_x, thread_block_sz_y);
  check_launch_error((cuda_thread_fun<<<nb,tpb>>>(n)));
  check_api_error(cudaDeviceSynchronize());
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_hello_2d cuda_hello_2d.cu

In [ ]:
%%sub
./cuda_hello_2d

* <font color="green">Exercise: Change the arguments of the following command line in various ways and see what happens</font>


In [ ]:
%%sub
./cuda_hello_2d 40 2 3

# 10. Data sharing between host (CPU) and device (GPU)
* GPU memory and CPU memory used to be logically separated
* That is, if you allocate memory on CPU and pass a _pointer_ to that memory region to GPU, GPU is not able to access it and caused a segmentation fault
* On GH200 (Miyabi's CPU-GPU integrated chip), this is no longer the case
  * all threads on GPU can access ordinary global variables/arrays, local variables/arrays, and memory allocated by ordinary malloc/mmap
* The following program demonstrates this


In [ ]:
%%writefile cuda_mem.cu
#include <assert.h>
#include <stdio.h>
#include <sys/mman.h>
#include <err.h>
#include "cuda_util.h"

__global__ void cuda_thread_fun(long * p, int n) {
  int i = blockDim.x * blockIdx.x + threadIdx.x;
  if (i < n) {
    p[i] = i * i;
  }
}

void print_array(long * p, long n) {
  printf("{");
  for (int i = 0; i < n; i++) {
    if (i > 0) printf(", ");
    printf("%ld", g[i]);
  }
  printf("}\n");
}

// global array
#define MAX_N 100
long g[MAX_N];

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 10);
  int thread_block_sz = (argc > 2 ? atoi(argv[2]) : 3);
  int n_thread_blocks = (n + thread_block_sz - 1) / thread_block_sz;
  assert(n <= MAX_N);
  // local array
  long l[MAX_N];
  // heap (new)
  long * h = new long[n];
  // heap (mmap)
  long * m = (long *)mmap(NULL, n * sizeof(long), PROT_READ | PROT_WRITE, MAP_ANONYMOUS | MAP_PRIVATE, -1, 0);
  if (m == MAP_FAILED) err(1, "mmap");
  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(g, n)));
  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(l, n)));
  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(h, n)));
  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(m, n)));
  check_api_error(cudaDeviceSynchronize());
  print_array(g, n);
  print_array(l, n);
  print_array(h, n);
  print_array(m, n);
  delete h;
  if (munmap(m, n * sizeof(long)) == -1) err(1, "munmap");
  return 0;
}

# <font color="green"> Problem 1 :  Executing a for loop with CUDA threads</font>
* Parallelize the following for loop with CUDA threads, assigning one CUDA thread to each iteration
* Use `__NVCC__` appropriately so it compiles both as a serial C++ program (by clang++) and CUDA (by nvcc)
```
  for (long i = 0; i < n; i++) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    record_logistic_map(a, tolerance, max_iter, &R[i]);
  }
```


In [ ]:
%%writefile cuda_for.cc
#include <err.h>
#include <math.h>
#include <sched.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>

/* return current time in nanoseconds */
long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

/* record structure for each parameter value */
typedef struct {
  long c;
  long t[2];
  int thread[2];
  int cpu[2];
} record_t;

/*
   Iterate x <- a*x*(1-x), starting from x0, until it converges
   sufficiently close to the stable fixed point x* = 1 - 1/a.
   Return the number of iterations, or max_iter if not converged.
*/
long steps_to_converge(double a, double tolerance, long max_iter) {
  double x = 0.1;
  for (long i = 1; i <= max_iter; i++) {
    double x_ = a * x * (1.0 - x);
    if (fabs(x - x_) < tolerance) {
      return i;
    }
    x = x_;
  }
  return max_iter;
}

void record_logistic_map(double a, double tolerance, long max_iter, record_t *R) {
  R->c = steps_to_converge(a, tolerance, max_iter);
}

void dump(record_t *R, double a_min, double a_max, long n, long t0, long t1) {
  // printf("T0=%ld T1=%ld\n", t0, t1);
  for (long i = 0; i < n; i++) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    printf("a=%f c=%ld\n", a, R[i].c);
  }
}

int main(int argc, char **argv) {
  int idx = 1;
  /* number of points to sample in the interval [a_min, a_max] */
  long n = (idx < argc ? atol(argv[idx]) : 1000); idx++;
  /* minimum/maximum value of a */
  double a_min = (idx < argc ? atof(argv[idx]) : 2.5); idx++;
  double a_max = (idx < argc ? atof(argv[idx]) : 2.999); idx++;
  /* |x_n+1 - x_n| < tolerance => converged */
  double tolerance = (idx < argc ? atof(argv[idx]) : 1.0e-11); idx++;
  /* quit after this number of iterations when not converged */
  long max_iter = (idx < argc ? atol(argv[idx]) : 10000); idx++;
  if (a_min < 0 || a_max <= a_min || max_iter <= 0) {
    errx(1, "expected 0 <= a_min < a_max and max_iter > 0");
  }

  record_t *R = (record_t *)calloc(n, sizeof(record_t));
  if (!R) err(1, "calloc");

  long t0 = cur_time_ns();
  for (long i = 0; i < n; i++) {
    double a = a_min + (a_max - a_min) * (i + 0.5) / n;
    record_logistic_map(a, tolerance, max_iter, &R[i]);
  }
  long t1 = cur_time_ns();

  dump(R, a_min, a_max, n, t0, t1);
  free(R);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -o cuda_for_cpp cuda_for.cc

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_for_cuda -x cu cuda_for.cc

In [ ]:
%%bash
./cuda_for_cpp > cuda_for_cpp.txt

In [ ]:
%%sub
./cuda_for_cuda > cuda_for_cuda.txt

In [ ]:
%%bash
ls -l cuda_for_cpp.txt cuda_for_cuda.txt
diff cuda_for_cpp.txt cuda_for_cuda.txt

# 11. Race condition and atomic operation 
* As threads launched in a single kernel call run concurrently, they are subject to the same race condition as OpenMP threads
* That is, if two threads access the same variable (or the same array element) and at least one of them is a write, there is a race and the program almost certainly has a bug

* In the following program, each thread increments a variable by one; it nevertheles does not print the number of threads launched and prints unpredictable results each time executed.


In [ ]:
%%writefile cuda_race.cu
#include <assert.h>
#include <stdio.h>

#include "cuda_util.h"

__global__ void cuda_thread_fun(unsigned long long * p, int n) {
  int i        = blockDim.x * blockIdx.x + threadIdx.x;
  if (i < n) {
    *p = *p + 1;
  }
}

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 1000);
  int thread_block_sz = (argc > 2 ? atoi(argv[2]) : 64);
  int n_thread_blocks = (n + thread_block_sz - 1) / thread_block_sz;

  unsigned long long c = 0;
  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(&c, n)));
  check_api_error(cudaDeviceSynchronize());
  printf("c = %lu\n", c);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_race cuda_race.cu

In [ ]:
%%sub
./cuda_race

<font color="green">Exercise: Execute the above program many times and observe the results; try changing parameters.</font>


In [ ]:
%%sub
./cuda_race 1000 64

* OpenMP had three basic tools --- critical, atomic and reduction --- to resolve race conditions depending on the situation.
* Roughly, CUDA only has an analogue to atomic, does not have an analogue to critical, and has a limited support for reduction.

## 11-1. Atomic add
* CUDA has
```
atomicAdd(T* p, T x);
```
function for various types of T.  
It performs `*p = *p + x` _atomically_, meaning that it is guaranteed that `*p` is not updated between the point `*p` is read and the point `*p` is written.

* See [atomicAdd](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#atomicadd) for reference

* <font color="green">Change the following program to resolve the race condition using `atomicAdd` and make sure the result always matches the number of threads launched.</font>


In [ ]:
%%writefile cuda_race_atomic_add.cu
#include <assert.h>
#include <stdio.h>

#include "cuda_util.h"

__global__ void cuda_thread_fun(unsigned long long * p, int n) {
  int i        = blockDim.x * blockIdx.x + threadIdx.x;
  if (i < n) {
    *p = *p + 1;
  }
}

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 1000);
  int thread_block_sz = (argc > 2 ? atoi(argv[2]) : 64);
  int n_thread_blocks = (n + thread_block_sz - 1) / thread_block_sz;

  unsigned long long c = 0;
  check_launch_error((cuda_thread_fun<<<n_thread_blocks,thread_block_sz>>>(&c, n)));
  check_api_error(cudaDeviceSynchronize());
  printf("c = %lu\n", c);
  return 0;
}

* To compile programs using `atomicAdd`, you need to give `--generate-code arch=compute_80,code=sm_80` to `nvcc`
* `--generate-code` specifies which GPU architectures/instruction set `nvcc` generates code for, so it might affect generated code in other ways including performance



In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc --generate-code arch=compute_80,code=sm_80 -o cuda_race_atomic_add cuda_race_atomic_add.cu

In [ ]:
%%sub
./cuda_race_atomic_add 10000 64
./cuda_race_atomic_add 100000 64

# 12. Barrier synchronization of threads
* Recent CUDA has the notion of cooperative groups, with which you can build a barrier synchronization between threads
* setup
```
#include <cooperative_groups.h>
namespace cg = cooperative_groups; // save typing
```
* create data representing a grouup
```
cg::grid_group g = cg::this_grid(); // all threads
```

* perform barrier synchronization when necessary (ensure no threads execute `<after>` until all threads finish `<before>`) 
```
  <before>
  g.sync();
  <after>
```

* You need to launch such kernels by
```
void * args[] = { a0, a1, ... };
cudaLaunchCooperativeKernel((void *)f, nb, bs, args);
```
instead of
```
f<<<nb,bs>>>(a0, a1, ...);
```

* See [Cooperative Groups](https://docs.nvidia.com/cuda/cuda-programming-guide/04-special-topics/cooperative-groups.html) for reference

# <font color="green"> Problem 2 :  Implement reduction on barrier synchronization</font>
Change the following program `sum_array()` so that it correctly outputs the sum of the array by implementing reduction on barrier synchronization.


In [ ]:
%%writefile cuda_sum.cu
#include <assert.h>
#include <stdio.h>

#include "cuda_util.h"

#include <cooperative_groups.h>

//using namespace cooperative_groups;
// Alternatively use an alias to avoid polluting the namespace with collective algorithms
namespace cg = cooperative_groups;

__global__ void sum_array(double * c, long n) {
  // should return c[0] + c[1] + ... + c[n-1] in c[0]
  // you can destroy other elements of the array
  cg::grid_group g = cg::this_grid();
  long i = g.thread_rank();
}

int main(int argc, char ** argv) {
  long n                = (argc > 1 ? atoi(argv[1]) : 10000);
  int threads_per_block = (argc > 2 ? atoi(argv[2]) : 64);
  int n_thread_blocks = (n + threads_per_block - 1) / threads_per_block;

  double * c = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) {
    c[i] = 1.0;
  }
  double * c_dev;
  void * args[2] = { (void *)&c, (void *)&n };
  check_api_error(cudaLaunchCooperativeKernel((void*)sum_array,
                                              n_thread_blocks,
                                              threads_per_block,
                                              args));
  check_api_error(cudaDeviceSynchronize());
  printf("sum = %f\n", c[0]);
  assert(c[0] == n);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -o cuda_sum cuda_sum.cu

In [ ]:
%%sub
./cuda_sum